# Facebook Ads Campaign Performance Analysis

This notebook analyses Facebook Ads campaign data to evaluate spend efficiency, ROMI (Return on Marketing Investment) trends, and the relationships between key advertising metrics.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (15, 5)

fad = pd.read_csv('h4_facebook_ads_data.csv')

## 1. Daily Ad Spend in 2021

Filtering the dataset to 2021 and visualising daily ad spend over time.

In [ ]:
y2021df = fad[fad['ad_date'].between('2021-01-01', '2021-12-31')].sort_values('ad_date')

y2021df.groupby('ad_date')['total_spend'].sum().plot(label='total spend', alpha=0.7)
plt.grid(True, linestyle='--', alpha=0.5)
plt.title('Daily Ads Spend in 2021', fontsize=18, fontweight='bold', alpha=0.85)
plt.xlabel('date', fontsize=14, fontweight='bold', alpha=0.7)
plt.ylabel('total spend', fontsize=14, fontweight='bold', alpha=0.7)
plt.legend()
plt.show()

## 2. Actual vs Rolling Average Spend

Applying a 7-day rolling average to smooth out daily fluctuations and reveal the underlying spend trend.

In [ ]:
daily_spend = y2021df.groupby('ad_date')['total_spend'].sum()
rolling_spend = daily_spend.rolling(window=7).mean()

plt.figure()
daily_spend.plot(label='actual', alpha=0.5)
rolling_spend.plot(label='rolling avg', linewidth=2)
plt.title('Actual vs Rolling Spend', fontsize=18, fontweight='bold', alpha=0.85)
plt.xlabel('date', fontsize=14, fontweight='bold', alpha=0.7)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

## 3. Daily ROMI (%)

Calculating Return on Marketing Investment as a percentage for each day.

In [ ]:
daily_romi = y2021df.groupby('ad_date').agg({'total_spend': 'sum', 'total_value': 'sum'})
daily_romi['ROMI'] = (daily_romi['total_value'] / daily_romi['total_spend'] * 100).round(2)

daily_romi['ROMI'].plot(alpha=0.7)
plt.title('Daily ROMI in %', fontsize=18, fontweight='bold', alpha=0.85)
plt.xlabel('date', fontsize=14, fontweight='bold', alpha=0.7)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

## 4. Rolling ROMI (%)

Applying a 7-day rolling sum before calculating ROMI, to reduce day-to-day noise and reveal the underlying efficiency trend.

In [ ]:
daily_romi_rolling = y2021df.groupby('ad_date').agg({'total_spend': 'sum', 'total_value': 'sum'}).rolling(7).sum()
daily_romi_rolling['rolling_ROMI'] = (daily_romi_rolling['total_value'] / daily_romi_rolling['total_spend'] * 100).round(2)

daily_romi_rolling['rolling_ROMI'].plot(alpha=0.7)
plt.title('Rolling ROMI in %', fontsize=18, fontweight='bold', alpha=0.85)
plt.xlabel('date', fontsize=14, fontweight='bold', alpha=0.7)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

## 5. Total Spend by Campaign

Comparing total ad spend across all campaigns.

In [ ]:
campaign_spend = fad.groupby('campaign_name')['total_spend'].sum().sort_values().plot(kind='barh', width=1, alpha=0.7)
plt.title('Total Spend by Campaign', fontsize=18, fontweight='bold', alpha=0.85)
plt.xlabel('total spend', fontsize=14, fontweight='bold', alpha=0.7)
plt.ylabel('')
plt.grid(True, linestyle='--', alpha=0.5)

for i in campaign_spend.containers:
    campaign_spend.bar_label(i, fmt='%.0f', padding=3)

plt.show()

## 6. ROMI (%) by Campaign

Comparing overall ROMI across campaigns to identify which ones deliver the best return relative to spend.

In [ ]:
camp_romi = fad.groupby('campaign_name')[['total_spend', 'total_value']].sum()
camp_romi['ROMI'] = (camp_romi['total_value'] / camp_romi['total_spend'] * 100).round(2)
camp_romi['ROMI'].sort_values().plot()

plt.title('ROMI % by Campaign', fontsize=18, fontweight='bold', alpha=0.85)
plt.xlabel('campaign name', fontsize=14, fontweight='bold', alpha=0.7)
plt.ylabel('ROMI %', fontsize=14, fontweight='bold', alpha=0.7)
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 7. Daily ROMI Range by Campaign

Using box plots to show the distribution and consistency of daily ROMI within each campaign, ordered by median ROMI.

In [ ]:
daily_romi_range = fad.groupby(['ad_date', 'campaign_name'])[['total_spend', 'total_value']].sum()
daily_romi_range['ROMI'] = (daily_romi_range['total_value'] / daily_romi_range['total_spend'] * 100).round(2)
daily_romi_range = daily_romi_range.reset_index()

orderby = daily_romi_range.groupby('campaign_name')['ROMI'].median().sort_values().index

sns.boxplot(data=daily_romi_range, x='campaign_name', y='ROMI', linewidth=1.5, order=orderby, boxprops=dict(alpha=0.5))
plt.title('Daily ROMI % Range by Campaign', fontsize=18, fontweight='bold', alpha=0.85)
plt.ylabel('ROMI %', fontsize=14, fontweight='bold', alpha=0.7)
plt.xlabel('')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 8. ROMI Distribution

A histogram showing the overall distribution of ROMI values across all campaign-days.

In [ ]:
ax = fad['romi'].plot(kind='hist', alpha=0.5)

for bar in ax.patches:
    height = bar.get_height()
    if height > 0:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height,
            f'{int(height)}',
            ha='center',
            va='bottom')

plt.title('ROMI Distribution', fontsize=18, fontweight='bold', alpha=0.85)
plt.ylabel('count', fontsize=14, fontweight='bold', alpha=0.7)
plt.xlabel('ROMI %')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 9. Correlation Between Key Metrics

Examining how spend, impressions, clicks, value, and efficiency metrics (CPC, CPM, CTR, ROMI) relate to one another.

In [ ]:
corr = fad[['total_spend', 'total_impressions', 'total_clicks', 'total_value', 'cpc', 'cpm', 'ctr', 'romi']].corr()

sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', center=0, alpha=0.9)
plt.title('Heatmap Correlation', fontsize=18, fontweight='bold', alpha=0.85)
plt.grid(False)
plt.show()

**Which metrics have the highest and lowest correlation?**

`total_value` has the strongest correlation with `total_spend`. `ROMI` has the weakest correlation with `total_spend` and the other metrics — meaning higher spend does not reliably predict a higher return.

**What does `total_value` correlate with?**

`total_value` correlates most strongly with `total_spend`, with a moderate relationship to `total_impressions` and `total_clicks` as well.

## 10. Spend vs Value Relationship

Visualising the relationship between total spend and total value with a regression line.

In [ ]:
sns.lmplot(x='total_spend', y='total_value', data=fad, height=5, aspect=3,
           scatter_kws={'s': 75, 'alpha': 0.5},
           line_kws={'color': 'red', 'linewidth': 2, 'alpha': 0.5})

plt.title('Spend vs Value Relationship', fontsize=20, fontweight='bold', alpha=0.85)
plt.xlabel('total spend', fontsize=14, fontweight='bold', alpha=0.7)
plt.ylabel('total value', fontsize=14, fontweight='bold', alpha=0.7)
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## Conclusions

- Total value tracks closely with total spend across campaigns, but ROMI does not scale with spend — some lower-spend campaigns achieve a far higher return per pound than higher-spend ones.
- Rolling averages reveal periods of sustained efficiency change that daily figures alone obscure, useful for spotting genuine trend shifts vs daily noise.
- The box plot comparison highlights which campaigns have consistent ROMI versus highly variable day-to-day performance, which is useful for identifying where budget reallocation could improve overall efficiency.